In [48]:
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio

from tqdm.auto import tqdm
from pathlib import Path

In [49]:
src_dir = (
    Path("~").expanduser()
    / "OneDrive - Stichting Deltares/PhD/Egypt/04_Data/2026_data/"
)
excel_path = (
    Path("~").expanduser()
    / "OneDrive - Stichting Deltares/PhD/Egypt_ERF_data/data_correlation.xlsx"
)


print (excel_path)


C:\Users\hermawan\OneDrive - Stichting Deltares\PhD\Egypt_ERF_data\data_correlation.xlsx


In [50]:
survey_df = pd.read_csv(src_dir.parent / "Result/ind_survey_data_gov_assigned.csv")
distribution_df = pd.read_csv(
    src_dir.parent / "Result/population_density_matrix.csv", index_col="NAME1_"
)
masking_df = pd.read_excel(
    src_dir.parent / "Result/correlation.xlsx", sheet_name="Sheet1"
)
excel_df = pd.read_excel(excel_path, sheet_name="command_unit2")

In [ ]:
col_normalize = [
    col for col in excel_df.columns
    if any(
        word in col.lower()
        for word in [
            "corrected",
            "hectares",
            "arable",
            "water_supply",
            "water_demand",
        ]
    )
    and col.lower() != "arable_km2_scaled"
]

excel_df[col_normalize] = excel_df[col_normalize].div(
    excel_df["arable_km2"],
    axis=0
)
excel_df.head()

NameError: name 'result_excel' is not defined

In [ ]:
print(f"Average survey weight: {survey_df['pweight'].mean()}")
print(f"Std of survey weight: {survey_df['pweight'].std()}")

In [ ]:
survey_df['sempinc'] = survey_df['sempinc'].fillna(0)
survey_df['irrgwag'] = survey_df['irrgwag'].fillna(0)
survey_df['totwag'] = survey_df['totwag'].fillna(0)

survey_df['totwag'] = survey_df['totwag'] + survey_df['sempinc'] + survey_df['irrgwag'] * 30

In [ ]:
female_df = survey_df[survey_df['sex_label'] == 'Female']
male_df = survey_df[survey_df['sex_label'] == 'Male']

In [ ]:
governorates = survey_df[survey_df["NAME1_"].notna()]["NAME1_"].unique()

rng = np.random.default_rng(seed=42)
sample_df = survey_df.copy(deep=True)
sample_df["command_unit"] = None
command_units = distribution_df.columns

gov_indices = {
    gov: idx.to_numpy() for gov, idx in survey_df.groupby("NAME1_").groups.items()
}
female_gov_indices = {
    gov: idx.to_numpy() for gov, idx in female_df.groupby("NAME1_").groups.items()
}
male_gov_indices = {
    gov: idx.to_numpy() for gov, idx in male_df.groupby("NAME1_").groups.items()
}

for gov, idx in gov_indices.items():
    probs = distribution_df.loc[gov].to_numpy()

    assignments = rng.choice(
        command_units,
        size=len(idx),
        p=probs,
    )

    sample_df.loc[idx, "command_unit"] = assignments

sample_df.loc[sample_df["command_unit"] == "other", "command_unit"] = None

sample_df = sample_df[sample_df["command_unit"].notna()]
final_sample = sample_df.merge(
    excel_df, left_on="command_unit", right_on="area", suffixes=("", "_spatial")
)

to_correlate = masking_df.loc[masking_df["correl"] == 1, "variable"]
cols = [c for c in to_correlate if c in final_sample.columns]

In [ ]:
survey_df['numwrk'].unique()

In [ ]:
n_assignments = 1000
female_corrs = []
male_corrs = []
for i in tqdm(range(n_assignments)):
    assignment_df = female_df.copy(deep=True)
    assignment_df["command_unit"] = None

    for gov, idx in female_gov_indices.items():
        probs = distribution_df.loc[gov].to_numpy()

        assignments = rng.choice(
            command_units,
            size=len(idx),
            p=probs,
        )

        assignment_df.loc[idx, "command_unit"] = assignments

    assignment_df.loc[assignment_df["command_unit"] == "other", "command_unit"] = None

    assignment_df = assignment_df[assignment_df["command_unit"].notna()]
    assignment_df = assignment_df.merge(
        excel_df, left_on="command_unit", right_on="area", suffixes=("", "_spatial")
    )

    cor_df = assignment_df[cols]
    cor_df["command_unit"] = cor_df["command_unit"].apply(
        lambda x: int(x.split("_")[-1])
    )

    corr_matrix = cor_df.corr(method="pearson")
    female_corrs.append(corr_matrix)

for i in tqdm(range(n_assignments)):
    assignment_df = male_df.copy(deep=True)
    assignment_df["command_unit"] = None

    for gov, idx in male_gov_indices.items():
        probs = distribution_df.loc[gov].to_numpy()

        assignments = rng.choice(
            command_units,
            size=len(idx),
            p=probs,
        )

        assignment_df.loc[idx, "command_unit"] = assignments

    assignment_df.loc[assignment_df["command_unit"] == "other", "command_unit"] = None

    assignment_df = assignment_df[assignment_df["command_unit"].notna()]
    assignment_df = assignment_df.merge(
        excel_df, left_on="command_unit", right_on="area", suffixes=("", "_spatial")
    )

    cor_df = assignment_df[cols]
    cor_df["command_unit"] = cor_df["command_unit"].apply(
        lambda x: int(x.split("_")[-1])
    )

    corr_matrix = cor_df.corr(method="pearson")
    male_corrs.append(corr_matrix)

In [ ]:
for col in cols:
    corr = np.corrcoef(
        assignment_df["pweight"],
        cor_df[col]
    )[0, 1]
    print(col, corr)

In [ ]:
female_corr_array = np.stack([c.to_numpy() for c in female_corrs])
male_corr_array = np.stack([c.to_numpy() for c in male_corrs])

cols_corr = female_corrs[0].columns

In [ ]:
for col in cols:
    print(
        col,
        assignment_df[col].nunique(dropna=True),
        assignment_df[col].isna().mean()
    )

In [ ]:
# avoid infinities for correlations exactly -1 or 1
female_corr_clipped = np.clip(female_corr_array, -0.999999, 0.999999)
male_corr_clipped = np.clip(male_corr_array, -0.999999, 0.999999)

female_z_array = np.arctanh(female_corr_clipped)
male_z_array = np.arctanh(male_corr_clipped)

female_mean_z = np.nanmean(female_z_array, axis=0)
male_mean_z = np.nanmean(male_z_array, axis=0)

female_mean_corr = np.tanh(female_mean_z)
male_mean_corr = np.tanh(male_mean_z)

female_mean_corr_df = pd.DataFrame(
    female_mean_corr,
    index=cols_corr,
    columns=cols_corr,
)

male_mean_corr_df = pd.DataFrame(
    male_mean_corr,
    index=cols_corr,
    columns=cols_corr,
)

In [ ]:
female_std_corr = np.nanstd(female_corr_array, axis=0)

female_std_corr_df = pd.DataFrame(
    female_std_corr,
    index=cols_corr,
    columns=cols_corr,
)

male_std_corr = np.nanstd(male_corr_array, axis=0)

male_std_corr_df = pd.DataFrame(
    male_std_corr,
    index=cols_corr,
    columns=cols_corr,
)

In [ ]:
female_p05_corr = np.nanpercentile(female_corr_array, 5, axis=0)
female_p50_corr = np.nanpercentile(female_corr_array, 50, axis=0)
female_p95_corr = np.nanpercentile(female_corr_array, 95, axis=0)

female_p05_corr_df = pd.DataFrame(female_p05_corr, index=cols_corr, columns=cols_corr)
female_p50_corr_df = pd.DataFrame(female_p50_corr, index=cols_corr, columns=cols_corr)
female_p95_corr_df = pd.DataFrame(female_p95_corr, index=cols_corr, columns=cols_corr)

male_p05_corr = np.nanpercentile(male_corr_array, 5, axis=0)
male_p50_corr = np.nanpercentile(male_corr_array, 50, axis=0)
male_p95_corr = np.nanpercentile(male_corr_array, 95, axis=0)

male_p05_corr_df = pd.DataFrame(male_p05_corr, index=cols_corr, columns=cols_corr)
male_p50_corr_df = pd.DataFrame(male_p50_corr, index=cols_corr, columns=cols_corr)
male_p95_corr_df = pd.DataFrame(male_p95_corr, index=cols_corr, columns=cols_corr)

In [ ]:
female_positive_fraction = np.nanmean(female_corr_array > 0, axis=0)
female_negative_fraction = np.nanmean(female_corr_array < 0, axis=0)

female_positive_fraction_df = pd.DataFrame(
    female_positive_fraction,
    index=cols_corr,
    columns=cols_corr,
)

female_negative_fraction_df = pd.DataFrame(
    female_negative_fraction,
    index=cols_corr,
    columns=cols_corr,
)

male_positive_fraction = np.nanmean(male_corr_array > 0, axis=0)
male_negative_fraction = np.nanmean(male_corr_array < 0, axis=0)

male_positive_fraction_df = pd.DataFrame(
    male_positive_fraction,
    index=cols_corr,
    columns=cols_corr,
)

male_negative_fraction_df = pd.DataFrame(
    male_negative_fraction,
    index=cols_corr,
    columns=cols_corr,
)

In [ ]:
female_summary_records = []

for row_var in cols_corr:
    for col_var in cols_corr:
        if row_var == col_var:
            continue

        i = cols_corr.get_loc(row_var)
        j = cols_corr.get_loc(col_var)

        female_summary_records.append(
            {
                "var_1": row_var,
                "var_2": col_var,
                "mean_r": female_mean_corr[i, j],
                "std_r": female_std_corr[i, j],
                "p05_r": female_p05_corr[i, j],
                "p50_r": female_p50_corr[i, j],
                "p95_r": female_p95_corr[i, j],
                "positive_fraction": female_positive_fraction[i, j],
                "negative_fraction": female_negative_fraction[i, j],
            }
        )

female_corr_summary_df = pd.DataFrame(female_summary_records)

male_summary_records = []

for row_var in cols_corr:
    for col_var in cols_corr:
        if row_var == col_var:
            continue

        i = cols_corr.get_loc(row_var)
        j = cols_corr.get_loc(col_var)

        male_summary_records.append(
            {
                "var_1": row_var,
                "var_2": col_var,
                "mean_r": male_mean_corr[i, j],
                "std_r": male_std_corr[i, j],
                "p05_r": male_p05_corr[i, j],
                "p50_r": male_p50_corr[i, j],
                "p95_r": male_p95_corr[i, j],
                "positive_fraction": male_positive_fraction[i, j],
                "negative_fraction": male_negative_fraction[i, j],
            }
        )

male_corr_summary_df = pd.DataFrame(male_summary_records)

In [ ]:
female_corr_summary_df.sort_values(by="std_r", ascending=False)

female_corr_summary_df.to_csv("female_corr_summary_sorted.csv", index=False)
male_corr_summary_df.to_csv("male_corr_summary_sorted.csv", index=False)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 10))
sns.heatmap(
    female_mean_corr_df,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
)
plt.show()

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(
    male_mean_corr_df,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
)
plt.show()

In [ ]:
female_corr_long = female_mean_corr_df.abs().stack().reset_index()

female_corr_long.columns = ["var1", "var2", "corr"]

female_corr_long = female_corr_long[female_corr_long["var1"] < female_corr_long["var2"]]

female_corr_long.sort_values("corr", ascending=False).head(40)

In [ ]:
male_corr_long = male_mean_corr_df.abs().stack().reset_index()

male_corr_long.columns = ["var1", "var2", "corr"]

male_corr_long = male_corr_long[male_corr_long["var1"] < male_corr_long["var2"]]

male_corr_long.sort_values("corr", ascending=False).head(40)